<a href="https://colab.research.google.com/github/DivyankBaluni/Data_Science/blob/main/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

You have been hired by a rookie movie producer to help him decide what type of movies to produce and which actors to cast. You have to back your recommendations based on thorough analysis of the data he shared with you which has the list of 3000 movies and the corresponding details.

 As a data scientist, you have to first explore the data and check its sanity.

Further, you have to answer the following questions:


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast

In [ ]:
df = pd.read_csv('imdb_data_1.csv')
print(df.shape)
print(df.head())

(3000, 23)
   id                              belongs_to_collection    budget  \
0   1  [{'id': 313576, 'name': 'Hot Tub Time Machine ...  14000000   
1   2  [{'id': 107674, 'name': 'The Princess Diaries ...  40000000   
2   3                                                NaN   3300000   
3   4                                                NaN   1200000   
4   5                                                NaN         0   

                                              genres  \
0                     [{'id': 35, 'name': 'Comedy'}]   
1  [{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...   
2                      [{'id': 18, 'name': 'Drama'}]   
3  [{'id': 53, 'name': 'Thriller'}, {'id': 18, 'n...   
4  [{'id': 28, 'name': 'Action'}, {'id': 53, 'nam...   

                            homepage    imdb_id original_language  \
0                                NaN  tt2637294                en   
1                                NaN  tt0368933                en   
2  http://sonyclassics.c

In [ ]:
print(df.columns.tolist())
df.columns = df.columns.str.capitalize()
# df.columns = df.columns.str.title()
print(df.columns.tolist())
print(df.dtypes)
print(df.isnull().sum())
print(df.duplicated().sum())
print(df.describe())

['id', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'imdb_id', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'Keywords', 'cast', 'crew', 'revenue']
['Id', 'Belongs_to_collection', 'Budget', 'Genres', 'Homepage', 'Imdb_id', 'Original_language', 'Original_title', 'Overview', 'Popularity', 'Poster_path', 'Production_companies', 'Production_countries', 'Release_date', 'Runtime', 'Spoken_languages', 'Status', 'Tagline', 'Title', 'Keywords', 'Cast', 'Crew', 'Revenue']
Id                         int64
Belongs_to_collection     object
Budget                     int64
Genres                    object
Homepage                  object
Imdb_id                   object
Original_language         object
Original_title            object
Overview                  object
Popularity               float64
Poster_path               object

In [ ]:
df['Budget'] = pd.to_numeric(df['Budget'], errors='coerce')
df['Revenue'] = pd.to_numeric(df['Revenue'], errors='coerce')
print((df['Budget'] == 0).sum())
print((df['Revenue'] == 0).sum())
clean = df[(df['Budget'] > 0) & (df['Revenue'] > 0)].copy()

812
0


In [ ]:
clean['Profit'] = clean['Revenue'] - clean['Budget']
clean['ROI']    = (clean['Profit'] / clean['Budget']) * 100
print(clean[['Title', 'Budget', 'Revenue', 'ROI', 'Profit']].head())

                                      Title    Budget   Revenue          ROI  \
0                    Hot Tub Time Machine 2  14000000  12314651   -12.038207   
1  The Princess Diaries 2: Royal Engagement  40000000  95149435   137.873588   
2                                  Whiplash   3300000  13092000   296.727273   
3                                   Kahaani   1200000  16000000  1233.333333   
5    Pinocchio and the Emperor of the Night   8000000   3261638   -59.229525   

     Profit  
0  -1685349  
1  55149435  
2   9792000  
3  14800000  
5  -4738362  


In [ ]:
def get_names(value) :
  if pd.isna(value) :
    return []
  try :
    items = ast.literal_eval(str(value))
    return [d['name'] for d in items if isinstance(d, dict) and 'name' in d]
  except:
    return []
def get_crew_member(value, job) :
  if pd.isna(value):
        return None
  try:
      items = ast.literal_eval(str(value))
      for person in items:
          if isinstance(person, dict) and person.get('job', '').lower() == job.lower():
              return person.get('name')
  except:
      pass
  return None

print(get_names(df['Genres'].iloc[0]))
print(get_crew_member(df['Crew'].iloc[0], 'Director'))

['Comedy']
Steve Pink


In [ ]:
clean['Director'] = clean['Crew'].apply(lambda x: get_crew_member(x, 'Director'))
clean['Producer'] = clean['Crew'].apply(lambda x: get_crew_member(x, 'Producer'))
print(clean[['Title', 'Director', 'Producer']].head(8))

                                       Title           Director  \
0                     Hot Tub Time Machine 2         Steve Pink   
1   The Princess Diaries 2: Royal Engagement     Garry Marshall   
2                                   Whiplash    Damien Chazelle   
3                                    Kahaani        Sujoy Ghosh   
5     Pinocchio and the Emperor of the Night     Hal Sutherland   
6                             The Possession       Ole Bornedal   
9                              A Mighty Wind  Christopher Guest   
10                                     Rocky   John G. Avildsen   

           Producer  
0      Andrew Panay  
1   Whitney Houston  
2   David Lancaster  
3       Sujoy Ghosh  
5              None  
6              None  
9      Karen Murphy  
10    Irwin Winkler  


Q1 : Which movie made the highest profit? Who were its producer and director? Identify the actors in that film.

In [ ]:
top_movie = clean.loc[clean['Profit'].idxmax()]
top_actors = get_names(top_movie['Cast'])[:5]
print(top_movie['Title'])
print(top_movie['Profit'])
print(top_movie['Director'])
print(top_movie['Producer'])
print(top_actors)

Furious 7
1316249360
James Wan
Vin Diesel
['Vin Diesel', 'Paul Walker', 'Dwayne Johnson', 'Michelle Rodriguez', 'Tyrese Gibson']


Q2 : Which language has the highest average ROI (return on investment)?

In [ ]:
lang_group = clean.groupby('Original_language')['ROI'].agg(['mean', 'count'])
lang_group.columns = ['Avg_ROI', 'Num_Movies']
lang_group = lang_group[lang_group['Num_Movies'] >= 5]
lang_group = lang_group.sort_values(by = 'Avg_ROI', ascending=False)
print(lang_group.head(10))
print("Best language : ", lang_group.index[0])
print(f"Average ROI : {lang_group.loc[lang_group.index[0], 'Avg_ROI'] :,.2f}%")

                        Avg_ROI  Num_Movies
Original_language                          
ko                 3.817941e+07          11
en                 5.439874e+05        1943
de                 4.349067e+02          11
zh                 4.084563e+02          14
es                 3.794217e+02          17
hi                 2.713313e+02          39
ta                 2.155159e+02          13
ja                 1.858405e+02          18
cn                 1.738661e+02           8
fr                 1.705117e+02          39
Best language :  ko
Average ROI : 38,179,410.23%


In [ ]:
lang_median = clean.groupby('Original_language')['ROI'].median()
lang_median = lang_median[clean.groupby('Original_language')['ROI'].count() >= 5]
print(lang_median.sort_values(ascending=False).head(5))

Original_language
cn    209.363617
ta    200.000000
zh    174.597727
ja    164.951639
hi    129.729730
Name: ROI, dtype: float64


Q3 : Find out the unique genres of movies in this dataset.

In [ ]:
df['Genre_list'] = df['Genres'].apply(get_names)
genre_series = df['Genre_list'].explode()
unique_genres = sorted(genre_series.dropna().unique())
genre_counts = genre_series.value_counts()
print(unique_genres)
print(genre_counts)

['Action', 'Adventure', 'Animation', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Foreign', 'History', 'Horror', 'Music', 'Mystery', 'Romance', 'Science Fiction', 'TV Movie', 'Thriller', 'War', 'Western']
Genre_list
Drama              1531
Comedy             1028
Thriller            789
Action              741
Romance             571
Crime               469
Adventure           439
Horror              301
Science Fiction     290
Family              260
Fantasy             232
Mystery             225
Animation           141
History             132
War                 100
Music               100
Documentary          87
Western              43
Foreign              31
TV Movie              1
Name: count, dtype: int64


Q4 : Make a table of all the producers and directors of each movie. Find the top 3 producers who have produced movies with the highest average RoI?

In [ ]:
pro_dir = clean[['Title', 'Director', 'Producer', 'ROI']].copy()
pro_dir.columns = ['Movie', 'Director', 'Producer', 'ROI(%)']
print(pro_dir.head(15).to_string(index=False))

producer_roi = (clean.dropna(subset=['Producer']).groupby('Producer')['ROI'].agg(['mean', 'count']))
producer_roi.columns = ['Avg_ROI', 'Num_Movies']
producer_roi = producer_roi[producer_roi['Num_Movies'] >= 3]  # min 3 movies
producer_roi = producer_roi.sort_values('Avg_ROI', ascending=False)

print(producer_roi.head(3))

                                   Movie          Director         Producer       ROI(%)
                  Hot Tub Time Machine 2        Steve Pink     Andrew Panay   -12.038207
The Princess Diaries 2: Royal Engagement    Garry Marshall  Whitney Houston   137.873588
                                Whiplash   Damien Chazelle  David Lancaster   296.727273
                                 Kahaani       Sujoy Ghosh      Sujoy Ghosh  1233.333333
  Pinocchio and the Emperor of the Night    Hal Sutherland             None   -59.229525
                          The Possession      Ole Bornedal             None   510.329107
                           A Mighty Wind Christopher Guest     Karen Murphy   212.504100
                                   Rocky  John G. Avildsen    Irwin Winkler 11623.514700
                         American Beauty        Sam Mendes      Bruce Cohen  2275.310673
                                 Be Cool      F. Gary Gray     Danny DeVito    79.671917
                     

Q5 : Which actor has acted in the most number of movies? Deep dive into the movies, genres and profits corresponding to this actor.

In [ ]:
clean['Actor_list'] = clean['Cast'].apply(lambda x: get_names(x)[:5])
all_actors = clean['Actor_list'].explode()
actor_movie_count = all_actors.value_counts()
top_actor = actor_movie_count.index[0]
top_count = actor_movie_count.iloc[0]
print(top_actor)
mask = clean['Actor_list'].apply(lambda actor_list: top_actor in actor_list)
actor_movies = clean[mask]
print(f"Average profit : {actor_movies['Profit'].mean() :,.2f}%")
print(f"Total profit earned  : {actor_movies['Profit'].sum():,.2f}%")

actor_genres = (actor_movies['Genres'].apply(get_names).explode().value_counts())
print(actor_movies[['Title', 'Profit', 'ROI']].sort_values('Profit', ascending=False).head(10).to_string(index=False))
print(actor_genres.to_string())

Robert De Niro
Average profit : 28,887,903.55%
Total profit earned  : 635,533,878.00%
           Title    Profit        ROI
Meet the Parents 275444045 500.807355
        Sleepers 121615285 276.398375
    Analyze This  96885658 121.107072
       Backdraft  77368585 103.158113
     Wag the Dog  49256513 328.376753
             Joy  41134059  68.556765
 The Deer Hunter  35000000 233.333333
    Men of Honor  16814909  52.546591
  Righteous Kill  13174566  21.957610
         Machete  11327899  56.639495
Genres
Drama       15
Thriller     9
Comedy       7
Crime        7
Action       5
Mystery      3
Romance      2
History      1
Horror       1
War          1


Q6 : Top 3 directors prefer which actors the most?

In [116]:
director_profit = (clean.dropna(subset=['Director']).groupby('Director')['Profit'].sum().sort_values(ascending=False))

top3_directors = director_profit.head(3).index.tolist()
for d in top3_directors:
  total = director_profit[d]
  print(f"  {d} : ${total/1e9:.2f}B")

print()
for director in top3_directors:
    dir_movies = clean[clean['Director'] == director]
    dir_actors = (dir_movies['Cast']
                  .apply(lambda x: get_names(x)[:5])
                  .explode()
                  .value_counts())

    print(f"{director} — Top 5 preferred actors:")
    print(dir_actors.head(5).to_string())
    print()

  Steven Spielberg : $3.48B
  Peter Jackson : $3.48B
  Michael Bay : $2.98B

Steven Spielberg — Top 5 preferred actors:
Cast
Harrison Ford       3
Richard Dreyfuss    2
Tom Hanks           2
Samantha Morton     1
Tom Cruise          1

Peter Jackson — Top 5 preferred actors:
Cast
Ian McKellen      4
Elijah Wood       2
Cate Blanchett    2
Orlando Bloom     2
Martin Freeman    2

Michael Bay — Top 5 preferred actors:
Cast
Josh Duhamel     4
Shia LaBeouf     3
Ed Harris        2
Tyrese Gibson    2
Megan Fox        2

